<a href="https://colab.research.google.com/github/Cristiand056/proyecto_sena_chatbot/blob/main/proyecto_sena_chatbot_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
import random
import json
import pickle
import numpy as np

import nltk
#from nltk.stem import WordNetLemmatizer
from nltk.stem import SnowballStemmer #Es mejor para el español
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Dropout


## **Preprocesamiento**

In [36]:
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')

stemmer = SnowballStemmer('spanish')

intents = json.loads(open('model/intents.json').read())

words = []      # vocabulario general
classes = []    # lista de intenciones (tags)
documents = []  # pares (palabras_del_patron, tag)
ignore_letters = ['?', '!', '.', ',']

def construir_vocabulario(intents):
  global words, classes, documents
  for intent in intents['intents']:
      for pattern in intent['patterns']:
          word_list = nltk.word_tokenize(pattern, language='spanish')
          words.extend(word_list)
          documents.append((word_list, intent['tag']))
          if intent['tag'] not in classes:
              classes.append(intent['tag'])

  # Lematiza (reduce palabras a su raíz) y elimina signos
  words = [stemmer.stem(w.lower()) for w in words if w not in ignore_letters]
  words = sorted(set(words))

  pickle.dump(words, open('model/words.pkl', 'wb'))
  pickle.dump(classes, open('model/classes.pkl', 'wb'))
  return words, classes, documents



[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [37]:
words, classes, documents = construir_vocabulario(intents)

## **Separar en train y test**

In [38]:
def bag_of_words(tokens_frase, vocabulario):
  tokens_stem = [stemmer.stem(w.lower()) for w in tokens_frase]
  bag = [1 if w in tokens_stem else 0 for w in vocabulario]
  return np.array(bag)

def preparar_datos_entrenamiento(documents, words, classes):
  training = []

  for tokens, tag in documents:
    entrada = bag_of_words(tokens, words)
    salida = [1 if c == tag else 0 for c in classes]

    training.append([entrada, salida])

  random.shuffle(training)
  training = np.array(training, dtype=object)

  train_x = list(training[:, 0])
  train_y = list(training[:, 1])
  return train_x, train_y

def construir_modelo(input_size, output_size):

  model = Sequential([
      Dense(128, input_shape=(input_size,), activation='relu'),
      Dropout(0.5),
      Dense(64, activation='relu'),
      Dropout(0.5),
      Dense(output_size, activation='softmax')
  ])

  model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
  return model


In [44]:
X, y = preparar_datos_entrenamiento(documents, words, classes)
X = np.array(X)
y = np.array(y)
modelo = construir_modelo(len(words), len(classes))
modelo.fit(X, y, epochs=100, batch_size=8, verbose=1)

# Guarda el modelo para usarlo en producción
modelo.save('model/chatbot_model.h5')

print("Entrenamiento completado. Modelo guardado.")

Epoch 1/100


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0206 - loss: 2.9322
Epoch 2/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1029 - loss: 2.8454
Epoch 3/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1399 - loss: 2.8050
Epoch 4/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1811 - loss: 2.7514
Epoch 5/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2181 - loss: 2.6846
Epoch 6/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1811 - loss: 2.5919
Epoch 7/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2140 - loss: 2.5064
Epoch 8/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2675 - loss: 2.3693
Epoch 9/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3251 - loss: 2.2513
Epoch 10/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3663 - loss: 2.1403
Epoch 11/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3909 - loss: 2.0467
Epoch 12/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4650 - lo

Entrenamiento completado. Modelo guardado.


In [24]:
from tensorflow.keras.models import load_model
import nltk
from nltk.stem import SnowballStemmer
import numpy as np
import pickle
import json
import random

In [45]:
stemmer    = SnowballStemmer('spanish')
modelo     = load_model('model/chatbot_model.h5')
words      = pickle.load(open('model/words.pkl', 'rb'))
classes    = pickle.load(open('model/classes.pkl', 'rb'))
intents    = json.load(open('model/intents.json', encoding='utf-8'))

UMBRAL_CONFIANZA = 0.75  # mínimo para dar una respuesta

def bag_of_words(frase: str) -> np.ndarray:
    # Tokeniza y lematiza el mensaje del usuario
    tokens = nltk.word_tokenize(frase.lower(), language='spanish')
    tokens_stem = [stemmer.stem(t) for t in tokens]
    bag = [1 if w in tokens_stem else 0 for w in words]
    return np.array(bag)

def predecir_intencion(frase: str) -> tuple[str, float]:
    # Convierte el mensaje en BoW y predice la intención
    bow = bag_of_words(frase)
    resultado = modelo.predict(np.array([bow]), verbose=0)[0]
    confianza = float(np.max(resultado))
    tag = classes[np.argmax(resultado)]
    return tag, confianza

def obtener_respuesta(frase: str) -> str:
    tag, confianza = predecir_intencion(frase)

    if confianza < UMBRAL_CONFIANZA:
        return "No entendí bien tu consulta. ¿Puedes reformularla o contactarnos directamente?"

    for intent in intents['intents']:
        if intent['tag'] == tag:
            return random.choice(intent['responses'])

    return "Ocurrió un error. Por favor contáctanos por WhatsApp."

In [46]:
# Frases que SÍ están en intents (debería acertar)
print(obtener_respuesta("tienen ron"))
print(obtener_respuesta("hola"))

# Frases que NO están pero son similares (aquí se ve la calidad real)
print(obtener_respuesta("me consiguen una botella de guaro"))
print(obtener_respuesta("quiero pedir unas cosas"))
print(obtener_respuesta("a qué horas abren"))
print(obtener_respuesta("se me olvidó la clave"))
print(obtener_respuesta("cuánto vale el marlboro"))

¡Claro! Manejamos licores nacionales como aguardiente y ron, e importados como whisky, tequila y vodka. ¿Qué necesitas?
Hola, soy el asistente virtual de Distribuidora X. ¿En qué te ayudo?
¡Claro! Manejamos licores nacionales como aguardiente y ron, e importados como whisky, tequila y vodka. ¿Qué necesitas?
Para hacer un pedido, inicia sesión y dirígete a la sección Crear Pedido. ¿Necesitas ayuda con algo específico?
Puedes contactarnos en horario hábil: lunes a sábado de 8am a 6pm.
Te enviaremos un enlace de recuperación al correo con el que te registraste.
Los precios varían según la referencia y cantidad. Revisa el catálogo o contáctanos para una cotización.
